# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a thorough walkthrough for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **DOI:** [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
- **URL:** https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and records
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata  # do NOT subscript or iterate, access as one object

# Print dataset-level information
print(f"Name: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"License: {getattr(metadata, 'license', '')}")


## 2. Data Overview
Review available record sets and their field `@id`s.

In [ ]:
# List all record sets available in the dataset (by '@id') and their fields.
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets explicitly defined in metadata. Attempting to autodiscover...")
    # Some datasets implicitly define a main RecordSet that contains primary data
    # mlcroissant's .record_sets may populate the list for compliant schemas

for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    fields = rs.get('fields', [])
    print("  Fields and their @id:")
    for field in fields:
        print(f"    - {field['@id']}: {field.get('name', '')} (dataType: {field.get('dataType', '')})")
    print('-----')

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis using their `@id`. Uses the record set and field `@id`s from the previous overview.


In [ ]:
# Let's get the record set @id for the main data table.

if not record_sets:
    raise RuntimeError("No record sets found in metadata! Check the dataset schema.")

main_record_set = record_sets[0]['@id']  # Usually the main table is the first one

print(f"Loading data from record set: {main_record_set}")
# List available fields/columns
main_fields = record_sets[0].get('fields', [])
print("Available fields:")
for field in main_fields:
    print(f"  {field['@id']}: {field.get('name', '')}")

# Load data for this RecordSet as a DataFrame
df = pd.DataFrame(dataset.records(record_set=main_record_set))
print("\nLoaded DataFrame shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We will apply common data operations, such as filtering records based on a numeric field, normalizing, and grouping. All field names are referenced by their `@id` as per mlcroissant schema. Please reference the field listing above to select meaningful variables.

In [ ]:
# Choose a numeric field by its @id (substitute with the correct @id from field listing; here's a common example for protein datasets)
numeric_field_id = None
# Try to find a numeric/int/float field
for field in main_fields:
    if str(field.get('dataType', '')).lower() in ('integer', 'float', 'number'):
        numeric_field_id = field['@id']
        break
if not numeric_field_id:
    # Fallback to a column that looks numeric
    for col in df.columns:
        if ('count' in col.lower() or 'mw' in col.lower() or 'abundan' in col.lower() or 'peptide' in col.lower() or 'score' in col.lower()):
            numeric_field_id = col
            break
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filter by a threshold (example: > 10)
threshold = 10
if numeric_field_id is not None and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id].apply(pd.to_numeric, errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce')
        - filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').mean()
    ) / filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    group_field_id = None
    for field in main_fields:
        if str(field.get('dataType', '')).lower() in ('text', 'string') and field['@id'] != numeric_field_id:
            group_field_id = field['@id']
            break
    if group_field_id and group_field_id in df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        )
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized values. Visualizations are created using `matplotlib` and `seaborn` for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns and len(filtered_df) > 0:
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (>{threshold})")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True, color='green')
    plt.title(f"Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.tight_layout()
    plt.show()
else:
    print('No data to plot.')

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and explore the FAIR² Mass Spectrometry dataset. We demonstrated how to:

- Load metadata and tabular data using schema `@id` references.
- Investigate record sets and available fields.
- Extract and filter records using numeric criteria.
- Normalize values and group by categorical labels.
- Visualize distributions for quick analysis.

Refer to the dataset metadata and Croissant definitions for further domain-specific processing or advanced ML workflows.